In [7]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import json
from io import StringIO
import statsmodels.api as sm #ARIMAX - https://people.duke.edu/~rnau/411arim.htm?utm_source=copilot.com # I plan to use p and q max of 5 based on the defaults in the r package auto.arima (https://pkg.robjhyndman.com/forecast/reference/auto.arima.html?utm_source=copilot.com)


In [8]:
# Preset alpha value for the Bernferroni correction
alpha = 0.05

In [ ]:
# See correlation_analysis_for_regression.ipynb for how these columns were selected
# Partially based on correlation and partially based on an understanding of the different meanings behind economic indicators
columns_to_keep = ['cpi_data_monthly', # SA
'rec_smooth_prob_data_monthly', # NSA
'unemployment_data_monthly', # SA
'interest_rate_mtg_data_weekly', # NSA
'consumer_sentiment_data_monthly', # NSA
'target' # Normalized by dividing closing price with cpi
]

In [10]:
# Parent directory for data loading
parent_dir = Path.cwd().parent.parent

In [11]:
# Import datasets and drop any nulls for clean processing
final_gig_m_df = pd.read_csv(os.path.join(parent_dir, 'data', 'final', 'gig_monthly_data.csv')).dropna().reset_index(drop=True)
final_sp500_m_df = pd.read_csv(os.path.join(parent_dir, 'data', 'final', 'sp500_monthly_data.csv')).dropna().reset_index(drop=True)

In [12]:
# Reduce datasets to only use columns to keep
final_gig_m_df = final_gig_m_df[columns_to_keep]
final_sp500_m_df = final_sp500_m_df[columns_to_keep]

In [ ]:
# In order to run the ARIMAX regression, I need to check for stationarity among all variables, including the endogenous variable (target)


In [19]:
# Get all possible combinations of exogenous variables for the regression analysis
# Since both datasets have the same columns, we can just use the columns from one of them to get the combinations
exog_cols = []
for c in final_gig_m_df.columns:
    temp_cols = []
    if c != 'target':
        temp_cols.append(c)
        for c2 in final_gig_m_df.columns:
            if c2 != 'target' and c2 != c:
                temp_cols.append(c2)
                temp_col_set = frozenset(sorted(temp_cols))
                exog_cols.append(temp_col_set)
exog_cols = frozenset(exog_cols)
exog_cols = [sorted(i) for i in exog_cols]

In [14]:
# Modify the alpha to use the bonferroni correction since we are testing many combinations
bonferroni_alpha = alpha / len(exog_cols)

In [ ]:
for exog_col in exog_cols:
    # Run regression analysis for gig dataset
    model_gig = sm.tsa.statespace.SARIMAX(final_gig_m_df['target'], exog=final_gig_m_df[exog_col], order=(5,0,5))
    results_gig = model_gig.fit()
    
    # Run regression analysis for sp500 dataset
    model_sp500 = sm.tsa.statespace.SARIMAX(final_sp500_m_df['target'], exog=final_sp500_m_df[exog_col], order=(5,0,5))
    results_sp500 = model_sp500.fit()